In [1]:
!nvidia-smi

Fri Oct 16 03:43:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   70C    P8             20W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))
print(whoami()["name"])

txchrixo


In [3]:
!pip install -q -U "transformers>=4.57" accelerate

In [4]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# MODEL_ID = "google/medgemma-1.5-4b-it"
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

a = torch.randn(256, 256, device="cuda", dtype=torch.bfloat16)
print("bfloat16 ok:", torch.isfinite((a @ a).float()).all().item())

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

bfloat16 ok: True


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

NVIDIA L4 8.9 GB used


In [11]:
import time

def ask(text, image=None, max_new_tokens=1500):
    content = [{"type": "text", "text": text}]
    if image is not None:
        content.append({"type": "image", "image": image})
    inputs = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    n = inputs["input_ids"].shape[-1]
    t = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    print(f"({time.time() - t:.0f} s, {len(out[0]) - n} tokens)")
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

print(ask("Réponds en une phrase : quelle est la capitale du Cameroun ?"))

(1 s, 13 tokens)
La capitale du Cameroun est Yaoundé.


In [6]:
import io
from google.colab import files
from PIL import Image

uploaded = files.upload()
images = {name: Image.open(io.BytesIO(data)).convert("RGB") for name, data in uploaded.items()}
print(list(images))

Saving R001_base.png to R001_base (1).png
['R001_base (1).png']


In [7]:
QUESTION = "Voici le résultat d'analyse de mon oncle. Qu'est-ce que ça veut dire ? Est-ce que c'est grave ?"

for name, img in images.items():
    print("=" * 80)
    print(name)
    print(ask(QUESTION, img))

R001_base (1).png
(45 s, 836 tokens)
Bonjour,

Voici l’interprétation de votre résultat d’analyse de PSA (Antigène prostatique spécifique) pour votre oncle Michel Fotsso, âgé de 52 ans.

---

### 📊 **Résultat : PSA total = 3,80 ng/mL**

**Valeur de référence : < 4,00 ng/mL**

➡️ **Interprétation :**

Le résultat **est dans la norme**. Le seuil de référence est inférieur à 4,00 ng/mL, et le résultat de votre oncle (3,80 ng/mL) est **inférieur à ce seuil**.

---

### 🧠 **Ce que signifie le PSA ?**

Le PSA est une protéine produite par la prostate. Il est utilisé comme **marqueur tumoral** pour détecter les cancers de la prostate, mais il n’est **pas spécifique** à la maladie. Il peut être légèrement élevé à cause de :

- Une **prostatite** (inflammation de la prostate)
- Une **hypertrophie prostatique** (augmentation de la taille de la prostate, très courante chez les hommes âgés)
- Une **maladie prostatique bénigne**
- Un **cancer de la prostate** (mais ce n’est pas le seul facteur)

--

In [8]:
name = list(images)[0]
print(ask("What is the PSA value on this report, with its unit?", images[name]))

(2 s, 17 tokens)
The PSA value on this report is **3.80 ng/mL**.


In [9]:
# Test A with the same comparison in text only, without the image
TEXT_QUESTIONS = [
    "Un PSA total de 3,80 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.",
    "A total PSA of 3.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.",
    "Un PSA total de 2,50 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.",
    "A total PSA of 0.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.",
]
for q in TEXT_QUESTIONS:
    print("Q:", q)
    print("A:", ask(q, max_new_tokens=100))
    print()

Q: Un PSA total de 3,80 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.
(2 s, 33 tokens)
A: Le PSA de 3,80 ng/mL est en dessous de la limite de référence (inférieure à 4,00 ng/mL).

Q: A total PSA of 3.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.
(1 s, 25 tokens)
A: The total PSA of 3.80 ng/mL is below the limit of 4.00 ng/mL.

Q: Un PSA total de 2,50 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.
(1 s, 29 tokens)
A: Le PSA de 2,50 ng/mL est en dessous de la limite de référence de 4,00 ng/mL.

Q: A total PSA of 0.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.
(2 s, 34 tokens)
A: A total PSA of 0.80 ng/mL is below the limit of 4.00 ng/mL, indicating it falls within the normal reference range.



In [10]:
# Test B with the image of R001 with a direct question
name = list(images)[0]
IMAGE_QUESTIONS = [
    "Le PSA total sur ce compte rendu est-il au-dessus ou en dessous de la valeur de référence ? Réponds en une phrase.",
    "Is the total PSA on this report above or below its reference range? Answer in one sentence.",
]
for q in IMAGE_QUESTIONS:
    print("Q:", q)
    print("A:", ask(q, images[name], max_new_tokens=100))
    print()

Q: Le PSA total sur ce compte rendu est-il au-dessus ou en dessous de la valeur de référence ? Réponds en une phrase.
(2 s, 14 tokens)
A: Le PSA total est en dessous de la valeur de référence.

Q: Is the total PSA on this report above or below its reference range? Answer in one sentence.
(3 s, 27 tokens)
A: The total PSA of 3.80 ng/mL is below its reference range of < 4.00 ng/mL.

